# Part 4: Báo Cáo Mã Hóa Biến Phân Loại & Chuẩn Hóa Đặc Trưng (Encoding & Normalization)
**Thành viên phụ trách:** Thành viên 4 | **Nhánh Git:** `feature/encoding-normalization`
---
Notebook thực hiện và kiểm định quy trình Tiền xử lý dữ liệu (Mã hóa One-Hot & Chuẩn hóa StandardScaler):
1. **Cô lập rò rỉ dữ liệu (Zero-leakage):** Loại bỏ hoàn toàn 4 cột rò rỉ (`delay_min`, `actual_time`, `status`, `status_detail`).
2. **Mã hóa OneHotEncoder:** Áp dụng `handle_unknown='ignore'` cho 5 biến phân loại (`airline_code`, `airport`, `direction`, `route_type`, `carrier_category`).
3. **Chuẩn hóa StandardScaler:** Đưa các biến số liên tục (`airport_load`, `precipitation`, `weather_hazard_score`, chu kỳ sin/cos giờ bay) về thang đo chuẩn N(0, 1).
4. **Giữ nguyên biến nhị phân:** Duy trì 14 cờ chỉ thị nhị phân (`is_peak_hour`, `is_rain`, `is_storm`, `is_golden_triangle`...).
5. **Đóng gói và lưu trữ Transformer:** Xuất `encoder.joblib`, `scaler.joblib`, `preprocessing_metadata.json` phục vụ suy luận mô hình.
*(Lưu ý: Phần chia tập dữ liệu Train/Test theo ngày được bàn giao cho Thành viên 5 thực hiện trong Part 5).*


### 1. Thực Thi Mã Hóa, Chuẩn Hóa & Kiểm Định Chống Rò Rỉ Dữ Liệu


In [ ]:
import pandas as pd
from modeling.encoding import (
    FlightDataPreprocessor, LEAKAGE_COLUMNS, CATEGORICAL_COLS,
    NUMERIC_CONTINUOUS_COLS, PASSTHROUGH_BINARY_COLS, PROCESSED_DIR
)
from features.features import run_feature_pipeline

# 1. Đọc dữ liệu đặc trưng từ Part 3
features_path = PROCESSED_DIR / 'flights_features.csv'
if not features_path.exists():
    df = run_feature_pipeline()
else:
    df = pd.read_csv(features_path)

print(f'Dữ liệu đầu vào: {df.shape[0]:,} dòng x {df.shape[1]} cột')

# 2. Khởi tạo và Fit Preprocessor
preprocessor = FlightDataPreprocessor()
preprocessor.fit(df)
X_transformed = preprocessor.transform(df)
X_df = preprocessor.to_dataframe(X_transformed)

print('\n=== KIỂM ĐỊNH RÒ RỈ DỮ LIỆU (ZERO-LEAKAGE CHECK) ===')
for col in LEAKAGE_COLUMNS:
    assert col not in X_df.columns, f'Cảnh báo: Cột rò rỉ {col} xuất hiện trong X!'
print('XÁC NHẬN: 100% không rò rỉ dữ liệu (delay_min, actual_time, status đã được loại bỏ).')

# 3. Lưu trữ các transformer artifacts phục vụ suy luận
preprocessor.save(PROCESSED_DIR)
print(f'\nĐã lưu trữ transformer thành công vào {PROCESSED_DIR}:')
print(f'- encoder.joblib ({len(preprocessor.encoded_cat_names)} cột danh mục sau OneHot)')
print(f'- scaler.joblib ({len(NUMERIC_CONTINUOUS_COLS)} cột chuẩn hóa StandardScaler)')
print(f'- preprocessing_metadata.json (Tổng số đặc trưng: {len(preprocessor.feature_names)})')
print(f'Kích thước ma trận đặc trưng hoàn chỉnh: {X_df.shape[0]:,} dòng x {X_df.shape[1]} cột')
